# Matura Grading Pipeline (Amazon Nova Pro)

Notebook testowy do automatycznego sprawdzania matury na podstawie: 
- arkusza z odpowiedziami ucznia: `data/exams/matura.pdf`
- schematow oceniania CKE: `data/exams/grading-schemas/*.pdf`

Implementacja spelnia wymagania: rozpoznanie sesji i poziomu, dobor schematu, ocenianie zamknietych zero-jedynkowo, ocenianie otwartych ze szczegolowym uzasadnieniem, punkty per zadanie i wynik koncowy.


## 1) Setup

Zainstaluj pakiety (poza notebookiem):

```bash
uv pip install -U boto3 python-dotenv pydantic pymupdf pillow
```

Wymagane zmienne srodowiskowe (np. w `.env`):

```dotenv
AWS_REGION=eu-central-1
BEDROCK_MODEL_ID=anthropic.claude-3-5-haiku-20241022-v1:0
# opcjonalnie, jesli konto wymaga inference profile:
BEDROCK_INFERENCE_PROFILE_ARN=
# alias wspierany wstecznie:
INFERENCE_PROFILE_ARN=
```


In [13]:
import io
import os
import re
import json
import time
import unicodedata
from pathlib import Path
from typing import Any

import boto3
import fitz
from PIL import Image
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import Markdown, display

load_dotenv()

AWS_REGION = os.getenv("AWS_REGION", "eu-central-1")
BEDROCK_MODEL_ID = os.getenv(
    "BEDROCK_MODEL_ID",
    os.getenv("NOVA_PRO_MODEL_ID", "anthropic.claude-3-5-haiku-20241022-v1:0"),
).strip()
BEDROCK_INFERENCE_PROFILE_ARN = os.getenv(
    "BEDROCK_INFERENCE_PROFILE_ARN",
    os.getenv("INFERENCE_PROFILE_ARN", os.getenv("NOVA_PRO_INFERENCE_PROFILE_ARN", "")),
).strip()
BEDROCK_EFFECTIVE_MODEL_ID = BEDROCK_INFERENCE_PROFILE_ARN or BEDROCK_MODEL_ID

PROJECT_ROOT = Path.cwd()
EXAM_PDF = PROJECT_ROOT / "data/exams/matura3.pdf"
SCHEMAS_DIR = PROJECT_ROOT / "data/exams/grading-schemas"

bedrock_runtime = boto3.client("bedrock-runtime", region_name=AWS_REGION)

print("AWS_REGION:", AWS_REGION)
print("BEDROCK_EFFECTIVE_MODEL_ID:", BEDROCK_EFFECTIVE_MODEL_ID)
print("EXAM_PDF exists:", EXAM_PDF.exists())
print("SCHEMAS_DIR exists:", SCHEMAS_DIR.exists())


AWS_REGION: eu-central-1
BEDROCK_EFFECTIVE_MODEL_ID: arn:aws:bedrock:eu-central-1:708919751779:inference-profile/eu.anthropic.claude-haiku-4-5-20251001-v1:0
EXAM_PDF exists: True
SCHEMAS_DIR exists: True


In [14]:
class ExamIdentification(BaseModel):
    level: str = Field(description="podstawa, rozszerzenie lub unknown")
    year: int = Field(description="Rok sesji, np. 2024")
    month: str = Field(description="Miesiac sesji, np. maj/czerwiec/marzec/grudzien")
    confidence: float = Field(ge=0.0, le=1.0)
    reasoning: str = Field(default="")

class TaskGrade(BaseModel):
    task_number: str
    task_type: str = Field(description="closed lub open")
    points_awarded: int
    points_max: int
    mistakes_explanation: str = Field(default="")

class GradingResult(BaseModel):
    exam: ExamIdentification
    tasks: list[TaskGrade]
    total_points_awarded: int
    total_points_max: int
    notes: str = Field(default="")


In [15]:
def normalize_token(text: str) -> str:
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def downscale_png_if_needed(png_bytes: bytes, max_side: int = 1800) -> bytes:
    image = Image.open(io.BytesIO(png_bytes)).convert("RGB")
    image.thumbnail((max_side, max_side))
    buffer = io.BytesIO()
    image.save(buffer, format="PNG", optimize=True)
    return buffer.getvalue()

def pdf_to_png_bytes(pdf_path: Path, dpi: int = 160, max_pages: int | None = None) -> list[bytes]:
    doc = fitz.open(pdf_path)
    page_count = len(doc) if max_pages is None else min(len(doc), max_pages)
    matrix = fitz.Matrix(dpi / 72.0, dpi / 72.0)

    images: list[bytes] = []
    for idx in range(page_count):
        page = doc[idx]
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        png_bytes = pix.tobytes("png")
        images.append(downscale_png_if_needed(png_bytes, max_side=1800))

    doc.close()
    return images

def safe_json_extract(text: str) -> Any:
    text = text.strip()
    if not text:
        raise ValueError("Pusta odpowiedz modelu.")

    # first try direct parse
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # then try to extract largest JSON object
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        raise ValueError("Nie znaleziono JSON w odpowiedzi modelu.")

    return json.loads(match.group(0))

def converse_json(image_bytes: list[bytes], prompt: str, max_tokens: int = 7000, temperature: float = 0.0) -> dict[str, Any]:
    content: list[dict[str, Any]] = []
    for img in image_bytes:
        content.append({"image": {"format": "png", "source": {"bytes": img}}})
    content.append({"text": prompt})

    t0 = time.time()
    response = bedrock_runtime.converse(
        modelId=BEDROCK_EFFECTIVE_MODEL_ID,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": temperature},
    )
    latency_s = time.time() - t0

    text_chunks = [
        c.get("text", "")
        for c in response.get("output", {}).get("message", {}).get("content", [])
        if "text" in c
    ]
    raw_text = "\n".join(chunk for chunk in text_chunks if chunk).strip()

    return {
        "raw_text": raw_text,
        "json": safe_json_extract(raw_text),
        "usage": response.get("usage", {}),
        "stop_reason": response.get("stopReason"),
        "latency_s": round(latency_s, 2),
    }


In [16]:
MONTH_ALIASES = {
    "maj": "maj",
    "czerwiec": "czerwiec",
    "marzec": "marzec",
    "grudzien": "grudzien",
}

def parse_schema_filename(path: Path) -> dict[str, Any] | None:
    # Example: 01 CKE 2024 Maj - ANS.pdf
    pattern = r"CKE\s+(\d{4})\s+(.+?)\s+-\s+ANS\.pdf$"
    m = re.search(pattern, path.name, flags=re.IGNORECASE)
    if not m:
        return None

    year = int(m.group(1))
    month_raw = m.group(2).strip()
    month_norm = normalize_token(month_raw)

    matched_month = None
    for alias, canonical in MONTH_ALIASES.items():
        if month_norm.startswith(alias):
            matched_month = canonical
            break

    if matched_month is None:
        matched_month = month_norm

    return {
        "path": path,
        "year": year,
        "month": matched_month,
        "display_name": path.name,
    }

def build_schema_index(schemas_dir: Path) -> list[dict[str, Any]]:
    rows: list[dict[str, Any]] = []
    for pdf_path in sorted(schemas_dir.glob("*.pdf")):
        parsed = parse_schema_filename(pdf_path)
        if parsed:
            rows.append(parsed)
    return rows

def select_schema_file(schema_index: list[dict[str, Any]], year: int, month: str) -> Path:
    month_norm = normalize_token(month)
    if month_norm in MONTH_ALIASES:
        month_norm = MONTH_ALIASES[month_norm]

    candidates = [
        row for row in schema_index
        if row["year"] == year and row["month"] == month_norm
    ]

    if not candidates:
        sessions = [f"{r['year']} {r['month']}" for r in schema_index]
        raise ValueError(
            f"Brak schematu dla sesji {year} {month_norm}. Dostepne: {sessions}"
        )

    return candidates[0]["path"]

schema_index = build_schema_index(SCHEMAS_DIR)
display(schema_index[:3])
print(f"Liczba dostepnych schematow: {len(schema_index)}")


[{'path': PosixPath('/Users/daniel/Developer/Projects/tutor-assistant/data/exams/grading-schemas/01 CKE 2024 Maj - ANS.pdf'),
  'year': 2024,
  'month': 'maj',
  'display_name': '01 CKE 2024 Maj - ANS.pdf'},
 {'path': PosixPath('/Users/daniel/Developer/Projects/tutor-assistant/data/exams/grading-schemas/02 CKE 2023 Maj - ANS.pdf'),
  'year': 2023,
  'month': 'maj',
  'display_name': '02 CKE 2023 Maj - ANS.pdf'},
 {'path': PosixPath('/Users/daniel/Developer/Projects/tutor-assistant/data/exams/grading-schemas/03 CKE 2023 Czerwiec - ANS.pdf'),
  'year': 2023,
  'month': 'czerwiec',
  'display_name': '03 CKE 2023 Czerwiec - ANS.pdf'}]

Liczba dostepnych schematow: 12


In [17]:
def identify_exam_session(exam_pdf: Path, schema_index: list[dict[str, Any]], preview_pages: int = 2) -> ExamIdentification:
    preview_images = pdf_to_png_bytes(exam_pdf, dpi=170, max_pages=preview_pages)
    sessions_hint = ", ".join(sorted({f"{r['year']} {r['month']}" for r in schema_index}))

    prompt = f"""
Jestes egzaminatorem CKE. Na podstawie pierwszej strony arkusza rozpoznaj sesje matury.
Zwroc TYLKO JSON (bez markdown):
{{
  "level": "podstawa" | "rozszerzenie" | "unknown",
  "year": <int>,
  "month": "maj" | "czerwiec" | "marzec" | "grudzien" | "unknown",
  "confidence": <float 0..1>,
  "reasoning": "krotkie uzasadnienie"
}}

Ustal poziom na podstawie pola "poziom" na pierwszej stronie.
Ustal sesje matury na podstawie pola "data" lub "termin" na pierwszej stronie.
Dostepne sesje kluczy w repo: {sessions_hint}
Jesli nie masz pewnosci miesiaca, zwroc najlepszy strzal + nizsze confidence.
"""

    result = converse_json(preview_images, prompt, max_tokens=1200, temperature=0.0)
    parsed = ExamIdentification.model_validate(result["json"])
    return parsed

exam_ident = identify_exam_session(EXAM_PDF, schema_index)
exam_ident


ExamIdentification(level='rozszerzenie', year=2023, month='czerwiec', confidence=0.95, reasoning="Arkusz zawiera 'Poziom rozszerzony' w nagłówku. Data egzaminu to '2 czerwca 2023 r.' co jednoznacznie wskazuje na sesję czerwcową 2023 roku.")

In [18]:
TASK_HEADER_RE = re.compile(r"zadanie\s*(\d+)", flags=re.IGNORECASE)
TASK_MAX_RE = re.compile(
    r"zadanie\s*(\d+)(?:[\.,]\d+)?\s*\(\s*0\s*[-–]\s*(\d+)\s*(?:pkt|punkty|punktow)?\s*\)",
    flags=re.IGNORECASE,
)


def extract_pdf_page_texts(pdf_path: Path, max_pages: int | None = None) -> list[str]:
    doc = fitz.open(pdf_path)
    page_count = len(doc) if max_pages is None else min(len(doc), max_pages)
    texts = [doc[idx].get_text("text") or "" for idx in range(page_count)]
    doc.close()
    return texts


def task_numbers_on_page(page_text: str) -> list[str]:
    return sorted(set(TASK_HEADER_RE.findall(page_text)), key=lambda x: int(x))


def build_task_page_spans(page_texts: list[str]) -> dict[str, tuple[int, int]]:
    task_pages: dict[str, list[int]] = {}
    active_task: str | None = None

    for page_idx, text in enumerate(page_texts):
        tasks = task_numbers_on_page(text)
        if tasks:
            active_task = tasks[0]
            for t in tasks:
                task_pages.setdefault(t, []).append(page_idx)
        elif active_task is not None:
            task_pages.setdefault(active_task, []).append(page_idx)

    spans: dict[str, tuple[int, int]] = {}
    for task_no, pages in task_pages.items():
        spans[task_no] = (min(pages), max(pages))
    return spans


def extract_task_max_points(page_texts: list[str]) -> dict[str, int]:
    result: dict[str, int] = {}
    for text in page_texts:
        for task_no, points_max in TASK_MAX_RE.findall(text or ""):
            result[str(task_no)] = int(points_max)
    return result


def split_pages_into_chunks(pages: list[int], max_chunk_pages: int) -> list[list[int]]:
    if not pages:
        return []

    chunks: list[list[int]] = []
    current: list[int] = [pages[0]]

    for page in pages[1:]:
        contiguous = page == current[-1] + 1
        if contiguous and len(current) < max_chunk_pages:
            current.append(page)
        else:
            chunks.append(current)
            current = [page]

    chunks.append(current)
    return chunks


def pdf_pages_to_png_bytes(pdf_path: Path, page_indices: list[int], dpi: int = 160) -> list[bytes]:
    doc = fitz.open(pdf_path)
    matrix = fitz.Matrix(dpi / 72.0, dpi / 72.0)
    images: list[bytes] = []

    for idx in page_indices:
        if idx < 0 or idx >= len(doc):
            continue
        page = doc[idx]
        pix = page.get_pixmap(matrix=matrix, alpha=False)
        png_bytes = pix.tobytes("png")
        images.append(downscale_png_if_needed(png_bytes, max_side=1800))

    doc.close()
    return images


def grade_chunk(
    exam_pdf: Path,
    schema_pdf: Path,
    exam_page_indices: list[int],
    schema_page_indices: list[int],
    exam_ident: ExamIdentification,
    mode: str,
    target_task_number: str | None = None,
    task_max_points: dict[str, int] | None = None,
) -> tuple[list[dict[str, Any]], dict[str, Any]]:
    exam_images = pdf_pages_to_png_bytes(exam_pdf, exam_page_indices, dpi=160)
    schema_images = pdf_pages_to_png_bytes(schema_pdf, schema_page_indices, dpi=160)
    images = exam_images + schema_images

    if mode == "closed":
        scope = "Oceniaj TYLKO zadania zamkniete widoczne na przeslanych stronach odpowiedzi ucznia."
    else:
        scope = f"Oceniaj TYLKO zadanie otwarte nr {target_task_number}. To zadanie moze byc rozlozone na wielu stronach."

    task_max_points = task_max_points or {}
    if mode == "open" and target_task_number and target_task_number in task_max_points:
        max_points_rule = (
            f"Dla zadania {target_task_number} points_max MUSI byc rowne {task_max_points[target_task_number]} "
            f"zgodnie z naglowkiem 'Zadanie {target_task_number} (0-{task_max_points[target_task_number]})'."
        )
    else:
        max_points_rule = (
            "Ustalaj points_max wylacznie z naglowka zadania w arkuszu/schemacie, np. 'Zadanie N (0-k)'. "
            "Nie wolno sumowac punktow z podpunktow ponad wartosc z naglowka."
        )

    prompt = f"""
Sprawdzasz arkusz maturalny z matematyki w trybie etapowym.
Masz dwa zestawy obrazow: najpierw odpowiedzi ucznia, potem fragment schematu oceniania CKE.

KRYTYCZNE ZASADY:
1) Potwierdz identyfikacje: level={exam_ident.level}, year={exam_ident.year}, month={exam_ident.month}.
2) {scope}
3) Zadanie zamkniete mozna zidentyfikowac po odpowiedziach A, B, C, D, ...
4) Zadania zamkniete oceniaj 0/1 i nie analizuj brudnopisu. Sprawdz tylko czy zaznaczona odpowiedz jest poprawna. Notatki cie nie interesuja.
5) Zadania otwarte oceniaj wg kryteriow CKE.
6) Nie zgaduj zadan niewidocznych w przekazanych obrazach.
7) Jesli w tym etapie nie ma zadnych ocenialnych zadan, zwroc pusta liste tasks oraz zera w totalach.
8) {max_points_rule}
9) W zadaniach otwartych: sam rysunek/szkic bez wymaganych obliczen, uzasadnienia i wnioskow = 0 punktow.
10) Nie myl rozwiazan ucznia z poprawnymi odpowiedziami.

FORMAT WYJSCIA - TYLKO JSON:
{{
  "exam": {{
    "level": "podstawa|rozszerzenie|unknown",
    "year": <int>,
    "month": "maj|czerwiec|marzec|grudzien|unknown",
    "confidence": <float 0..1>,
    "reasoning": "string"
  }},
  "tasks": [
    {{
      "task_number": "string",
      "task_type": "closed|open",
      "points_awarded": <integer>,
      "points_max": <integer>,
      "mistakes_explanation": ""
    }}
  ],
  "total_points_awarded": <integer>,
  "total_points_max": <integer>,
  "notes": "krotkie uwagi"
}}

Dodatkowe reguly:
- Dla task_type=closed pole mistakes_explanation ma byc pustym stringiem.
- Dla task_type=open: jesli brak bledow, mistakes_explanation ma byc pusty string.
- Dla task_type=open: jesli points_awarded == points_max, mistakes_explanation musi byc pusty string.
- Dla task_type=open: jesli mistakes_explanation nie jest puste, points_awarded musi byc mniejsze od points_max.
- Nie dodawaj tekstu poza JSON.
"""

    result = converse_json(images, prompt, max_tokens=2500, temperature=0.0)
    parsed = GradingResult.model_validate(result["json"])
    parsed_payload = parsed.model_dump()
    return parsed_payload["tasks"], {
        "latency_s": result["latency_s"],
        "usage": result["usage"],
        "stop_reason": result["stop_reason"],
        "exam_pages": exam_page_indices,
        "schema_pages": schema_page_indices,
        "mode": mode,
        "target_task_number": target_task_number,
        "prompt": prompt,
        "raw_text": result["raw_text"],
        "model_json": result["json"],
        "parsed_tasks": parsed_payload["tasks"],
    }


def base_task_number(task_number: str) -> str:
    m = re.search(r"\d+", str(task_number))
    return m.group(0) if m else str(task_number).strip()


def normalize_task_number(task_number: str) -> str:
    value = str(task_number).strip().replace(",", ".")
    value = re.sub(r"\s+", "", value)
    return value


def local_task_sort_key(task_number: str) -> tuple[int, str]:
    m = re.search(r"\d+", str(task_number))
    return (int(m.group(0)) if m else 10**9, str(task_number))


def merge_tasks(tasks: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_task: dict[str, dict[str, Any]] = {}

    for task in tasks:
        task_no_raw = str(task.get("task_number", "")).strip()
        task_no = normalize_task_number(task_no_raw)
        if not task_no:
            continue

        normalized_task = dict(task)
        normalized_task["task_number"] = task_no

        prev = by_task.get(task_no)
        if prev is None:
            by_task[task_no] = normalized_task
            continue

        prev_score = int(prev.get("points_awarded", 0))
        new_score = int(normalized_task.get("points_awarded", 0))
        if new_score > prev_score:
            by_task[task_no] = normalized_task

    return sorted(by_task.values(), key=lambda t: local_task_sort_key(t["task_number"]))


def grade_matura_exam(
    exam_pdf: Path,
    schema_pdf: Path,
    exam_ident: ExamIdentification,
    exam_pages_limit: int | None = None,
    schema_pages_limit: int | None = None,
) -> dict[str, Any]:
    exam_page_texts = extract_pdf_page_texts(exam_pdf, max_pages=exam_pages_limit)
    schema_page_texts = extract_pdf_page_texts(schema_pdf, max_pages=schema_pages_limit)

    exam_task_spans = build_task_page_spans(exam_page_texts)
    schema_task_spans = build_task_page_spans(schema_page_texts)

    exam_task_max = extract_task_max_points(exam_page_texts)
    schema_task_max = extract_task_max_points(schema_page_texts)
    task_max_points = {**exam_task_max, **schema_task_max}

    all_exam_pages = list(range(len(exam_page_texts)))
    open_task_numbers = sorted(exam_task_spans.keys(), key=lambda x: int(x))

    open_pages: set[int] = set()
    for task_no in open_task_numbers:
        start, end = exam_task_spans[task_no]
        # Kazde zadanie otwarte (takze jedno-stronicowe) wycina strony z puli closed.
        open_pages.update(range(start, end + 1))

    pages_with_task_header = sorted(
        [idx for idx, text in enumerate(exam_page_texts) if task_numbers_on_page(text)]
    )
    closed_pages = sorted([p for p in pages_with_task_header if p not in open_pages])
    closed_chunks = split_pages_into_chunks(closed_pages, max_chunk_pages=2)

    all_tasks: list[dict[str, Any]] = []
    chunk_meta: list[dict[str, Any]] = []

    for exam_chunk in closed_chunks:
        schema_chunk = list(range(min(2, len(schema_page_texts))))
        tasks, meta = grade_chunk(
            exam_pdf=exam_pdf,
            schema_pdf=schema_pdf,
            exam_page_indices=exam_chunk,
            schema_page_indices=schema_chunk,
            exam_ident=exam_ident,
            mode="closed",
            task_max_points=task_max_points,
        )
        all_tasks.extend([t for t in tasks if t.get("task_type") == "closed"])
        chunk_meta.append(meta)

    for task_no in open_task_numbers:
        start, end = exam_task_spans[task_no]
        exam_chunk = list(range(start, end + 1))

        if task_no in schema_task_spans:
            s_start, s_end = schema_task_spans[task_no]
            schema_chunk = list(range(s_start, s_end + 1))
        else:
            schema_chunk = list(range(min(3, len(schema_page_texts))))

        tasks, meta = grade_chunk(
            exam_pdf=exam_pdf,
            schema_pdf=schema_pdf,
            exam_page_indices=exam_chunk,
            schema_page_indices=schema_chunk,
            exam_ident=exam_ident,
            mode="open",
            target_task_number=task_no,
            task_max_points=task_max_points,
        )
        matched_tasks = [
            t for t in tasks
            if base_task_number(str(t.get("task_number", "")).strip()) == task_no
        ]
        if not matched_tasks:
            matched_tasks = [{
                "task_number": task_no,
                "task_type": "open",
                "points_awarded": 0,
                "points_max": int(task_max_points.get(task_no, 0)),
                "mistakes_explanation": "Brak poprawnej odpowiedzi modelu dla tego zadania w etapie open.",
            }]
        all_tasks.extend(matched_tasks)
        chunk_meta.append(meta)

    tasks_final = merge_tasks(all_tasks)

    corrected_tasks: list[dict[str, Any]] = []
    for task in tasks_final:
        task_no = str(task.get("task_number", "")).strip()
        corrected = dict(task)
        if task_no in task_max_points:
            corrected_max = int(task_max_points[task_no])
            corrected["points_max"] = corrected_max
            corrected["points_awarded"] = min(int(corrected.get("points_awarded", 0)), corrected_max)

        points_awarded = int(corrected.get("points_awarded", 0))
        points_max = int(corrected.get("points_max", 0))
        explanation = str(corrected.get("mistakes_explanation", "")).strip()

        if corrected.get("task_type") == "open":
            if explanation and points_awarded >= points_max and points_max > 0:
                corrected["points_awarded"] = max(0, points_max - 1)
                points_awarded = int(corrected["points_awarded"])
            if points_awarded >= points_max and points_max > 0:
                corrected["mistakes_explanation"] = ""

        corrected_tasks.append(corrected)

    tasks_final = corrected_tasks
    total_awarded = sum(int(t.get("points_awarded", 0)) for t in tasks_final)
    total_max = sum(int(t.get("points_max", 0)) for t in tasks_final)

    payload = {
        "exam": exam_ident.model_dump(),
        "tasks": tasks_final,
        "total_points_awarded": total_awarded,
        "total_points_max": total_max,
        "notes": "Etapowe ocenianie: closed<=2 strony, open=1 zadanie.",
        "_meta": {
            "schema_file": str(schema_pdf),
            "chunk_count": len(chunk_meta),
            "chunk_meta": chunk_meta,
            "exam_task_spans": exam_task_spans,
            "schema_task_spans": schema_task_spans,
            "task_max_points": task_max_points,
        },
    }

    parsed = GradingResult.model_validate(payload)
    validated = parsed.model_dump()
    validated["_meta"] = payload["_meta"]
    return validated


selected_schema = select_schema_file(schema_index, exam_ident.year, exam_ident.month)
print("Selected schema:", selected_schema.name)

grading_result = grade_matura_exam(
    exam_pdf=EXAM_PDF,
    schema_pdf=selected_schema,
    exam_ident=exam_ident,
    exam_pages_limit=None,
    schema_pages_limit=None,
)

grading_result["_meta"]


Selected schema: 03 CKE 2023 Czerwiec - ANS.pdf


{'schema_file': '/Users/daniel/Developer/Projects/tutor-assistant/data/exams/grading-schemas/03 CKE 2023 Czerwiec - ANS.pdf',
 'chunk_count': 13,
 'chunk_meta': [{'latency_s': 3.79,
   'usage': {'inputTokens': 3877,
    'outputTokens': 335,
    'totalTokens': 4212,
    'cacheReadInputTokens': 0,
    'cacheWriteInputTokens': 0},
   'stop_reason': 'end_turn',
   'exam_pages': [3],
   'schema_pages': [1],
   'mode': 'open',
   'target_task_number': '1',
   'prompt': '\nSprawdzasz arkusz maturalny z matematyki w trybie etapowym.\nMasz dwa zestawy obrazow: najpierw odpowiedzi ucznia, potem fragment schematu oceniania CKE.\n\nKRYTYCZNE ZASADY:\n1) Potwierdz identyfikacje: level=rozszerzenie, year=2023, month=czerwiec.\n2) Oceniaj TYLKO zadanie otwarte nr 1. To zadanie moze byc rozlozone na wielu stronach.\n3) Zadanie zamkniete mozna zidentyfikowac po odpowiedziach A, B, C, D, ...\n4) Zadania zamkniete oceniaj 0/1 i nie analizuj brudnopisu. Sprawdz tylko czy zaznaczona odpowiedz jest poprawna

In [19]:
def task_sort_key(task_number: str) -> tuple[int, str]:
    m = re.search(r"\d+", str(task_number))
    return (int(m.group(0)) if m else 10**9, str(task_number))

def render_report_markdown(result: dict[str, Any]) -> str:
    exam = result["exam"]
    tasks = sorted(result["tasks"], key=lambda t: task_sort_key(t["task_number"]))

    lines: list[str] = []
    lines.append("## Raport oceniania matury")
    lines.append("")
    lines.append(
        f"Arkusz: **{exam['level']}**, sesja **{exam['year']} {exam['month']}** (confidence: {exam['confidence']:.2f})"
    )
    lines.append(f"Schemat: `{Path(result['_meta']['schema_file']).name}`")
    lines.append("")

    lines.append("### Zadania zamkniete")
    closed_found = False
    for task in tasks:
        if task["task_type"] != "closed":
            continue
        closed_found = True
        lines.append(
            f"- Zadanie {task['task_number']}: {task['points_awarded']} / {task['points_max']} pkt"
        )
    if not closed_found:
        lines.append("- Brak zadan zamknietych wykrytych przez model.")

    lines.append("")
    lines.append("### Zadania otwarte")
    open_found = False
    for task in tasks:
        if task["task_type"] != "open":
            continue
        open_found = True
        explanation = task.get("mistakes_explanation", "").strip()
        if explanation:
            lines.append(
                f"- Zadanie {task['task_number']}: {task['points_awarded']} / {task['points_max']} pkt. Bledy: {explanation}"
            )
        else:
            lines.append(
                f"- Zadanie {task['task_number']}: {task['points_awarded']} / {task['points_max']} pkt"
            )
    if not open_found:
        lines.append("- Brak zadan otwartych wykrytych przez model.")

    lines.append("")
    lines.append(
        f"## Wynik koncowy: **{result['total_points_awarded']} / {result['total_points_max']} pkt**"
    )

    return "\n".join(lines)

report_md = render_report_markdown(grading_result)
display(Markdown(report_md))

# opcjonalnie: zapis surowego wyniku
output_path = PROJECT_ROOT / "outputs/matura_grading_result.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
with output_path.open("w", encoding="utf-8") as f:
    json.dump(grading_result, f, ensure_ascii=False, indent=2)
print("Saved:", output_path)


## Raport oceniania matury

Arkusz: **rozszerzenie**, sesja **2023 czerwiec** (confidence: 0.95)
Schemat: `03 CKE 2023 Czerwiec - ANS.pdf`

### Zadania zamkniete
- Brak zadan zamknietych wykrytych przez model.

### Zadania otwarte
- Zadanie 1: 2 / 2 pkt
- Zadanie 2: 3 / 3 pkt
- Zadanie 3: 3 / 3 pkt
- Zadanie 4: 3 / 3 pkt
- Zadanie 5: 3 / 3 pkt
- Zadanie 6: 3 / 3 pkt
- Zadanie 7: 4 / 4 pkt
- Zadanie 8: 4 / 4 pkt
- Zadanie 9: 4 / 4 pkt
- Zadanie 10: 5 / 5 pkt
- Zadanie 11: 5 / 5 pkt
- Zadanie 12: 5 / 5 pkt
- Zadanie 13: 6 / 6 pkt

## Wynik koncowy: **50 / 50 pkt**

Saved: /Users/daniel/Developer/Projects/tutor-assistant/outputs/matura_grading_result.json


In [20]:
grading_result

{'exam': {'level': 'rozszerzenie',
  'year': 2023,
  'month': 'czerwiec',
  'confidence': 0.95,
  'reasoning': "Arkusz zawiera 'Poziom rozszerzony' w nagłówku. Data egzaminu to '2 czerwca 2023 r.' co jednoznacznie wskazuje na sesję czerwcową 2023 roku."},
 'tasks': [{'task_number': '1',
   'task_type': 'open',
   'points_awarded': 2,
   'points_max': 2,
   'mistakes_explanation': ''},
  {'task_number': '2',
   'task_type': 'open',
   'points_awarded': 3,
   'points_max': 3,
   'mistakes_explanation': ''},
  {'task_number': '3',
   'task_type': 'open',
   'points_awarded': 3,
   'points_max': 3,
   'mistakes_explanation': ''},
  {'task_number': '4',
   'task_type': 'open',
   'points_awarded': 3,
   'points_max': 3,
   'mistakes_explanation': ''},
  {'task_number': '5',
   'task_type': 'open',
   'points_awarded': 3,
   'points_max': 3,
   'mistakes_explanation': ''},
  {'task_number': '6',
   'task_type': 'open',
   'points_awarded': 3,
   'points_max': 3,
   'mistakes_explanation': ''